# Marsa Matruh & Qattara Depression — Elevation & Land Use Analysis

**Study Area:** Marsa Matruh Governorate + Qattara Depression, NW Egypt  
**DEM:** Copernicus GLO-30 (30 m) via Google Earth Engine  
**Land Use:** ESA WorldCover v200 2021 (10 m)  

### What this notebook does
1. Authenticates with Google Earth Engine
2. Loads and clips the Copernicus DEM to the study area
3. Computes elevation statistics (min, max, mean, std dev)
4. Locates the highest and lowest elevation points
5. Loads ESA WorldCover land use / land cover data
6. Displays an interactive map with all layers and legends
7. Optionally exports rasters to Google Drive

### Key feature
The **Qattara Depression** (expected lowest point ≈ −133 m) is the lowest point in Africa  
and is clearly visible in this region.

---
> **How to run:** Go to `Runtime → Run all` (or press `Ctrl+F9`)  
> You will be prompted to authenticate with Google once in Cell 2.

## Cell 1 — Install Dependencies
Installs `earthengine-api` and `geemap` if not already present in the Colab environment.

In [ ]:
# Install required packages
# This only needs to run once per Colab session
!pip install -q earthengine-api geemap

print('Packages installed successfully.')

## Cell 2 — Authentication & Initialization

`ee.Authenticate()` will open a browser popup asking you to sign in with your Google account  
that has an approved Earth Engine project.  

**Replace `'ee-your-project-id'`** with your actual GEE Cloud project ID  
(visible at [console.cloud.google.com](https://console.cloud.google.com)).

In [ ]:
import ee
import geemap

# ─── EDIT THIS: replace with your GEE Cloud project ID ────────────────────────
GEE_PROJECT = 'ee-your-project-id'
# ──────────────────────────────────────────────────────────────────────────────

try:
    ee.Initialize(project=GEE_PROJECT)
    print('Earth Engine initialized successfully.')
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print('Authenticated and initialized.')

## Cell 3 — Configuration
Edit these values to customize the analysis.

In [ ]:
CONFIG = {
    'contour_interval': 50,                          # Contour line spacing (m)
    'stats_scale':      90,                          # Scale for statistics (m)
                                                     # 90 m = fast; 30 m = precise but slow
    'export_to_drive':  False,                       # Set True to export GeoTIFFs to Drive
    'export_folder':    'GEE_MarsaMatruh',           # Google Drive folder name
    'export_dem':       'MarsaMatruh_Qattara_DEM',   # DEM export filename (no extension)
    'export_lulc':      'MarsaMatruh_Qattara_LULC',  # Land use export filename
    'export_scale':     30,                          # Export resolution (m)
}

print('Configuration loaded.')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

## Cell 4 — Study Area

Defines the region of interest covering **Marsa Matruh Governorate** and the **Qattara Depression**.  
You can swap in any of the commented alternatives below.

In [ ]:
# Default: Full Marsa Matruh Governorate + Qattara Depression
#   West  : 24.8 E  (Libya border)
#   East  : 30.5 E  (eastern extent)
#   South : 28.2 N  (southern Qattara)
#   North : 31.6 N  (Mediterranean coast)
geometry = ee.Geometry.Polygon(
    [[[24.8, 28.2],   # SW corner  [lon, lat]
      [30.5, 28.2],   # SE corner
      [30.5, 31.6],   # NE corner
      [24.8, 31.6]]], # NW corner
    None, False
)

# ── Alternative areas (uncomment one to use instead) ──────────────────────
# Qattara Depression only (tight bbox)
# geometry = ee.Geometry.Rectangle([26.0, 28.5, 29.8, 30.8])

# Marsa Matruh city and coast only
# geometry = ee.Geometry.Rectangle([26.5, 30.5, 28.5, 31.5])
# ──────────────────────────────────────────────────────────────────────────

area_km2 = geometry.area(maxError=100).divide(1e6).getInfo()
print(f'Study area defined.  Total area: {area_km2:,.0f} km²')

## Cell 5 — Color Palettes

In [ ]:
# Elevation palette: deep blue (lowest) → white (sea level) → dark red (highest)
# White is anchored near 0 m to highlight the Qattara Depression's below-sea-level floor
ELEV_PALETTE = [
    '08306b',  # very deep blue  (lowest / below sea level)
    '2171b5',  # blue
    '6baed6',  # light blue
    'bdd7e7',  # very pale blue
    'ffffff',  # white           (sea level 0 m)
    'ffffb2',  # pale yellow
    'fecc5c',  # yellow-orange
    'fd8d3c',  # orange
    'f03b20',  # red-orange
    'bd0026',  # red
    '67000d',  # dark red        (highest peaks)
]

# ESA WorldCover land cover palette (official ESA colors, ordered by class value)
LC_PALETTE = [
    '006400',  # 10  Tree cover
    'ffbb22',  # 20  Shrubland
    'ffff4c',  # 30  Grassland
    'f096ff',  # 40  Cropland
    'fa0000',  # 50  Built-up
    'b4b4b4',  # 60  Bare / sparse vegetation  <-- dominant in this desert region
    'f0f0f0',  # 70  Snow and ice
    '0064c8',  # 80  Permanent water bodies
    '0096a0',  # 90  Herbaceous wetland
    '00cf75',  # 95  Mangroves
    'fae6a0',  # 100 Moss and lichen
]

# Land cover class definitions (used to build the legend)
LC_CLASSES = [
    {'value': 10,  'label': 'Tree cover',         'color': '006400'},
    {'value': 20,  'label': 'Shrubland',          'color': 'ffbb22'},
    {'value': 30,  'label': 'Grassland',          'color': 'ffff4c'},
    {'value': 40,  'label': 'Cropland',           'color': 'f096ff'},
    {'value': 50,  'label': 'Built-up',           'color': 'fa0000'},
    {'value': 60,  'label': 'Bare / sparse veg.', 'color': 'b4b4b4'},
    {'value': 70,  'label': 'Snow and ice',       'color': 'f0f0f0'},
    {'value': 80,  'label': 'Permanent water',    'color': '0064c8'},
    {'value': 90,  'label': 'Herbaceous wetland', 'color': '0096a0'},
    {'value': 95,  'label': 'Mangroves',          'color': '00cf75'},
    {'value': 100, 'label': 'Moss and lichen',    'color': 'fae6a0'},
]

print('Color palettes defined.')

## Cell 6 — Load Copernicus DEM GLO-30

Mosaics multiple DEM tiles covering the study area and clips to the geometry.

In [ ]:
cop_dem = (
    ee.ImageCollection('COPERNICUS/DEM/GLO30')
    .filterBounds(geometry)
    .select('DEM')
    .mosaic()
    .clip(geometry)
)

print('Copernicus GLO-30 DEM loaded.')
print('Band names:', cop_dem.bandNames().getInfo())
print('Projection:', cop_dem.projection().getInfo())

## Cell 7 — Elevation Statistics

Computes min, max, mean, and standard deviation over the study area.  
Uses 90 m scale for speed (set `stats_scale: 30` in CONFIG for full precision).

In [ ]:
stats = cop_dem.reduceRegion(
    reducer=(
        ee.Reducer.minMax()
        .combine(ee.Reducer.mean(),   sharedInputs=True)
        .combine(ee.Reducer.stdDev(), sharedInputs=True)
    ),
    geometry=geometry,
    scale=CONFIG['stats_scale'],
    maxPixels=1e13,
    bestEffort=True,
)

# Resolve server-side computation to Python floats (synchronous)
stats_result = stats.getInfo()
min_val  = stats_result['DEM_min']
max_val  = stats_result['DEM_max']
mean_val = stats_result['DEM_mean']
std_val  = stats_result['DEM_stdDev']

print(f'Statistics computed at {CONFIG["stats_scale"]} m scale.')
print(f'  Min elevation :  {min_val:.1f} m')
print(f'  Max elevation :  {max_val:.1f} m')
print(f'  Mean elevation:  {mean_val:.1f} m')
print(f'  Std deviation :  {std_val:.1f} m')

## Cell 8 — Locate Highest & Lowest Points

Masks pixels matching the global min/max elevation value,  
then extracts their centroid coordinates.

In [ ]:
max_elev = ee.Number(stats.get('DEM_max'))
min_elev = ee.Number(stats.get('DEM_min'))

# Highest point
highest_point = (
    cop_dem.eq(max_elev.toFloat()).selfMask()
    .reduceToVectors(
        geometry=geometry,
        scale=CONFIG['stats_scale'],
        geometryType='centroid',
        maxPixels=1e13,
        bestEffort=True,
    )
)

# Lowest point
lowest_point = (
    cop_dem.eq(min_elev.toFloat()).selfMask()
    .reduceToVectors(
        geometry=geometry,
        scale=CONFIG['stats_scale'],
        geometryType='centroid',
        maxPixels=1e13,
        bestEffort=True,
    )
)

# Resolve coordinates to Python lists
highest_coords = ee.Feature(highest_point.first()).geometry().coordinates().getInfo()
lowest_coords  = ee.Feature(lowest_point.first()).geometry().coordinates().getInfo()

print('======================================================')
print('  ELEVATION ANALYSIS RESULTS')
print('======================================================')
print(f'  Study area:     {area_km2:,.0f} km²')
print()
print('  HIGHEST POINT')
print(f'    Elevation:    {max_val:.1f} m')
print(f'    Longitude:    {highest_coords[0]:.5f} E')
print(f'    Latitude:     {highest_coords[1]:.5f} N')
print()
print('  LOWEST POINT  (expect approx. -133 m in Qattara Depression)')
print(f'    Elevation:    {min_val:.1f} m')
print(f'    Longitude:    {lowest_coords[0]:.5f} E')
print(f'    Latitude:     {lowest_coords[1]:.5f} N')
print('======================================================')

## Cell 9 — Precompute Terrain Layers

These are computed server-side as `ee.Image` objects  
and passed to the map later.

In [ ]:
contour_interval = CONFIG['contour_interval']

# Contour lines: pixels within 8 m of a contour level
contours = (
    cop_dem
    .divide(contour_interval).floor().multiply(contour_interval)
    .subtract(cop_dem).abs().lt(8)
    .selfMask()
)

# Study area boundary outline
boundary_img = ee.Image().paint(
    featureCollection=ee.FeatureCollection([ee.Feature(geometry)]),
    color=1,
    width=2,
)

# Below-sea-level mask (highlights Qattara Depression floor)
sea_level_mask = cop_dem.lt(0).selfMask()

print('Terrain layers computed (server-side).')
print(f'  Contour interval: {contour_interval} m')

## Cell 10 — Load Land Use / Land Cover

**Dataset:** ESA WorldCover v200 (2021)  
**Asset ID:** `ESA/WorldCover/v200`  
**Resolution:** 10 m global coverage

| Class | Value | Expected in this region |
|---|---|---|
| Bare / sparse veg. | 60 | Dominant — desert interior & Qattara floor |
| Cropland | 40 | Coastal strip & Nile Delta fringe |
| Built-up | 50 | Marsa Matruh city |
| Permanent water | 80 | Coastal lagoons |

In [ ]:
world_cover = (
    ee.ImageCollection('ESA/WorldCover/v200')
    .filterBounds(geometry)
    .first()
    .clip(geometry)
)

print('ESA WorldCover 2021 (10 m) loaded.')
print('Band names:', world_cover.bandNames().getInfo())

## Cell 11 — Interactive Map

Displays all layers using **geemap** (interactive, zoom and toggle layers).  
The land use layer is **hidden by default** — toggle it on in the Layer Control (top-right of map).

> **Note:** If the map does not render, run `Cell 1` again to reinstall `geemap`,  
> then restart the runtime (`Runtime → Restart runtime`) and re-run all cells.

In [ ]:
# Create the map and center on the study area
Map = geemap.Map()
Map.centerObject(geometry, zoom=7)

# ── Layer 1: Elevation (color-coded DEM) ─────────────────────────────────────
Map.addLayer(
    cop_dem,
    {'min': min_val, 'max': max_val, 'palette': ELEV_PALETTE},
    'Elevation — Copernicus GLO-30',
    shown=True,
    opacity=1.0,
)

# ── Layer 2: Contour lines ────────────────────────────────────────────────────
Map.addLayer(
    contours,
    {'palette': ['222222']},
    f'Contour Lines ({contour_interval} m)',
    shown=True,
    opacity=0.5,
)

# ── Layer 3: Study area boundary ──────────────────────────────────────────────
Map.addLayer(
    boundary_img,
    {'palette': ['FFFFFF']},
    'Study Area Boundary',
    shown=True,
)

# ── Layer 4: Highest point (red marker) ───────────────────────────────────────
Map.addLayer(
    highest_point,
    {'color': 'FF3300'},
    f'Highest Point  ({max_val:.0f} m)',
    shown=True,
)

# ── Layer 5: Lowest point (blue marker) ───────────────────────────────────────
Map.addLayer(
    lowest_point,
    {'color': '0099FF'},
    f'Lowest Point — Qattara  ({min_val:.0f} m)',
    shown=True,
)

# ── Layer 6: Below sea level overlay ──────────────────────────────────────────
Map.addLayer(
    sea_level_mask,
    {'palette': ['003366'], 'opacity': 0.35},
    'Below Sea Level (< 0 m)',
    shown=True,
)

# ── Layer 7: Land use / land cover (hidden by default) ────────────────────────
Map.addLayer(
    world_cover,
    {'min': 10, 'max': 100, 'palette': LC_PALETTE},
    'Land Use / Land Cover (ESA WorldCover 2021)',
    shown=False,   # Toggle on in Layer Control to view
)

# ── Land cover legend ─────────────────────────────────────────────────────────
lc_legend_dict = {cls['label']: '#' + cls['color'] for cls in LC_CLASSES}
Map.add_legend(
    title='Land Use / Land Cover',
    legend_dict=lc_legend_dict,
    position='bottomright',
)

# ── Elevation colorbar ────────────────────────────────────────────────────────
Map.add_colorbar(
    vis_params={'min': min_val, 'max': max_val, 'palette': ELEV_PALETTE},
    label=f'Elevation (m)  —  {min_val:.0f} m (Qattara) to {max_val:.0f} m',
    orientation='horizontal',
    position='bottomleft',
    transparent_bg=True,
)

# Display the map
Map

## Cell 12 — (Optional) Export to Google Drive

Set `CONFIG['export_to_drive'] = True` in **Cell 3** and re-run from Cell 3.  
Export tasks run on GEE servers — monitor them at  
[code.earthengine.google.com](https://code.earthengine.google.com/) (Tasks tab)  
or by calling `ee.batch.Task.list()` in the next cell.

In [ ]:
if CONFIG['export_to_drive']:

    # Export DEM (Copernicus GLO-30, clipped to study area)
    task_dem = ee.batch.Export.image.toDrive(
        image=cop_dem,
        description=CONFIG['export_dem'],
        folder=CONFIG['export_folder'],
        fileNamePrefix=CONFIG['export_dem'],
        region=geometry,
        scale=CONFIG['export_scale'],
        crs='EPSG:4326',
        maxPixels=1e13,
        fileFormat='GeoTIFF',
    )
    task_dem.start()
    print(f'DEM export started:       {CONFIG["export_dem"]}')
    print(f'  Status: {task_dem.status()["state"]}')

    # Export land use (ESA WorldCover 2021)
    # Use scale=10 for full 10 m resolution (large file); 30 m is practical
    task_lulc = ee.batch.Export.image.toDrive(
        image=world_cover,
        description=CONFIG['export_lulc'],
        folder=CONFIG['export_folder'],
        fileNamePrefix=CONFIG['export_lulc'],
        region=geometry,
        scale=30,
        crs='EPSG:4326',
        maxPixels=1e13,
        fileFormat='GeoTIFF',
    )
    task_lulc.start()
    print(f'Land use export started:  {CONFIG["export_lulc"]}')
    print(f'  Status: {task_lulc.status()["state"]}')

    print()
    print('Both tasks submitted to GEE. Files will appear in your Google Drive')
    print(f'under the folder: {CONFIG["export_folder"]}')

else:
    print('Export is disabled.  Set CONFIG["export_to_drive"] = True in Cell 3 to enable.')

## Cell 13 — (Optional) Check Export Task Status

In [ ]:
# Lists all active and recent GEE export tasks for your account
tasks = ee.batch.Task.list()
if tasks:
    print(f'{'Task ID':<30} {'State':<15} {'Description'}')
    print('-' * 70)
    for t in tasks[:10]:   # show latest 10
        s = t.status()
        print(f"{s.get('id','')[:28]:<30} {s.get('state',''):<15} {s.get('description','')}")
else:
    print('No active export tasks found.')